In [1]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
llm = ChatOllama(model="gemma4:e2b", base_url="http://127.0.0.1:11434")

llm.invoke([HumanMessage("잘 지냈어?")])

AIMessage(content='네, 저는 잘 지냈습니다. 😊\n\n당신은요? 잘 지내고 계신가요?', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-18T10:32:57.8032686Z', 'done': True, 'done_reason': 'stop', 'total_duration': 25072114800, 'load_duration': 24595275600, 'prompt_eval_count': 21, 'prompt_eval_duration': 144671000, 'eval_count': 25, 'eval_duration': 326879000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0b413-34d9-71f2-bacf-004035094163-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 21, 'output_tokens': 25, 'total_tokens': 46})

In [2]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time


In [3]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [4]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

# (6) 생성된 llm 답변 출력
print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-18T10:33:24.8446128Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4274205000, 'load_duration': 9375600, 'prompt_eval_count': 157, 'prompt_eval_duration': 886709000, 'eval_count': 321, 'eval_duration': 3370631000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0b413-efb6-7491-8031-e04761bea80d-0', tool_calls=[{'name': 'get_current_time', 'args': {'location': 'Busan', 'timezone': 'Asia/Seoul'}, 'id': 'a88245cd-c208-417e-9ca6-ee1a65dc084c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 157, 'output_tokens': 321, 'total_tokens': 478})]


In [5]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'location': 'Busan', 'timezone': 'Asia/Seoul'}
Asia/Seoul (Busan) 현재시각 2026-09-18 19:33:25 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-18T10:33:24.8446128Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4274205000, 'load_duration': 9375600, 'prompt_eval_count': 157, 'prompt_eval_duration': 886709000, 'eval_count': 321, 'eval_duration': 3370631000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0b413-efb6-7491-8031-e04761bea80d-0', tool_calls=[{'name': 'get_current_time', 'args': {'location': 'Busan', 'timezone': 'Asia/Seoul'}, 'id': 'a88245cd-c208-417e-9ca6-ee1a65dc084c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 157, 'output_tokens': 321, 'total_tokens': 478}),
 ToolMessage(content='Asia/Seoul (Busan) 현재시각 2026-09-18 19:33:25 ', name

In [6]:
llm_with_tools.invoke(messages)

AIMessage(content='부산은 현재 **2026년 9월 18일 19시 33분 25초**입니다.', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-18T10:33:28.273117Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1507950400, 'load_duration': 8270800, 'prompt_eval_count': 227, 'prompt_eval_duration': 149078000, 'eval_count': 137, 'eval_duration': 1341181000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0b414-07ec-70e1-8cf5-4d70883d44db-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 227, 'output_tokens': 137, 'total_tokens': 364})